In [1]:
%pip install torch scikit-learn transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import torch
import logging
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification

Model_DIR = "./model"
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("NER")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {DEVICE}")

2025-10-24 17:56:50,654 - INFO - Using device: cpu


In [4]:

def _clean_entity_text(text: str, strip_outer: bool = False) -> str:
    """
    Clean up spaces and punctuation in extracted entity text.
    - Remove extra spaces around punctuation
    - Collapse multiple spaces
    - Optionally strip surrounding punctuation (e.g., "(GPL)." -> "GPL")
    """
    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,;:!?()])", r"\1", text)
    # Remove spaces after opening and before closing brackets
    text = re.sub(r"(\()\s+", r"\1", text)
    text = re.sub(r"\s+(\))", r"\1", text)
    # Collapse multiple spaces
    text = re.sub(r"\s{2,}", " ", text)
    text = text.strip()

    if strip_outer:
        text = re.sub(r"^[\(\[\{]+", "", text)
        text = re.sub(r"[\)\]\}\.\,]+$", "", text)
        text = text.strip()

    return text

class NERPipeline:
    def __init__(self, model_dir: str, threshold: float = 0.7, strip_outer_punct: bool = False):
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to(DEVICE)
        self.model.eval()
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id
        self.threshold = threshold
        self.strip_outer_punct = strip_outer_punct

    @torch.no_grad()
    def __call__(self, text: str):
        # Tokenize input
        enc = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            return_offsets_mapping=True
        )
        offset_mapping = enc.pop("offset_mapping").tolist()
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        # Forward pass
        outputs = self.model(**enc)

        # ✅ Compute probabilities and apply threshold
        probs = F.softmax(outputs.logits, dim=2)[0]
        confidences, predictions = torch.max(probs, dim=1)
        confidences = confidences.cpu().tolist()
        predictions = predictions.cpu().tolist()

        for i, conf in enumerate(confidences):
            if conf < self.threshold:
                predictions[i] = self.label2id["O"]

        tokens = enc["input_ids"][0].cpu().tolist()
        entities = []
        current_entity = None
        current_confidences = []

        # 🧠 Merge entity spans
        for token_id, pred_id, (start, end), conf in zip(tokens, predictions, offset_mapping[0], confidences):
            token_str = self.tokenizer.convert_ids_to_tokens(token_id)
            raw_token = token_str
            clean_token = token_str.replace("##", "")
            label = self.id2label[pred_id]

            # Skip special tokens
            if raw_token in ["[CLS]", "[SEP]", "[PAD]"]:
                continue

            if label == "O":
                if current_entity:
                    current_entity["confidence"] = float(sum(current_confidences) / len(current_confidences))
                    current_entity["text"] = _clean_entity_text(current_entity["text"], self.strip_outer_punct)
                    entities.append(current_entity)
                    current_entity = None
                    current_confidences = []
                continue

            # Merge with previous if same label and contiguous
            if current_entity and current_entity["label"] == label and (start == current_entity["end"] or start == current_entity["end"] + 1):
                if raw_token.startswith("##"):
                    current_entity["text"] += clean_token
                else:
                    current_entity["text"] += " " + clean_token
                current_entity["end"] = end
                current_confidences.append(conf)
            else:
                if current_entity:
                    current_entity["confidence"] = float(sum(current_confidences) / len(current_confidences))
                    current_entity["text"] = _clean_entity_text(current_entity["text"], self.strip_outer_punct)
                    entities.append(current_entity)
                current_entity = {
                    "text": clean_token,
                    "label": label,
                    "start": start,
                    "end": end
                }
                current_confidences = [conf]


        # finalize last entity
        if current_entity:
            current_entity["confidence"] = float(sum(current_confidences) / len(current_confidences))
            current_entity["text"] = _clean_entity_text(current_entity["text"], self.strip_outer_punct)
            entities.append(current_entity)

        return entities

In [7]:
pipeline = NERPipeline(Model_DIR)
text = """If you use this package in your work, cite us as below.

@article{Loetgering:23,
author = {Lars Loetgering and Mengqi Du and Dirk Boonzajer Flaes and Tomas Aidukas and Felix Wechsler and Daniel S. Penagos Molina and Max Rose and Antonios Pelekanidis and Wilhelm Eschen and J\"{u}rgen Hess and Thomas Wilhein and Rainer Heintzmann and Jan Rothhardt and Stefan Witte},
journal = {Opt. Express},
number = {9},
pages = {13763--13797},
publisher = {Optica Publishing Group},
title = {PtyLab.m/py/jl: a cross-platform, open-source inverse modeling toolbox for conventional and Fourier ptychography},
volume = {31},
month = {Apr},
year = {2023},
doi = {10.1364/OE.485370},
}

This software requires Python 3.8+ and is licensed under MIT License.
For installation instructions, see https://github.com/example/repo.
The software runs on Linux and Windows operating systems.
"""
result = pipeline(text)
logger.info(result)

2025-10-24 17:58:50,762 - INFO - [{'text': '@ article { Loetgering: 23, author = { Lars Loetgering and Mengqi Du and Dirk Boonzajer Flaes and Tomas Aidukas and Felix Wechsler and Daniel S. Penagos Molina and Max Rose and Antonios Pelekanidis and Wilhelm Eschen and J " { u } rgen Hess and Thomas Wilhein and Rainer Heintzmann and Jan Rothhardt and Stefan Witte }, journal = { Opt. Express }, number = { 9 }, pages = { 13763 - - 13797 }, publisher = { Optica Publishing Group }, title = { PtyLab. m / py / jl: a cross - platform, open - source inverse modeling toolbox for conventional and Fourier ptychography }, volume = { 31 }, month = { Apr }, year = { 2023 }, doi = { 10. 1364 / OE. 485370 }, }', 'label': 'REFERENCE_PUBLICATION', 'start': 57, 'end': 666, 'confidence': 0.9992987450194244}, {'text': 'Python 3. 8 +', 'label': 'RUNTIME_PLATFORM', 'start': 691, 'end': 702, 'confidence': 0.9855436563491822}, {'text': 'MIT License', 'label': 'LICENSE', 'start': 725, 'end': 736, 'confidence': 0.987